In [ ]:
# Colab setup - run this first. Installs m3 plus the tutorial plotting extras.
# (Colab already ships PyTorch, which m3 uses as its engine.)
%pip install -q "git+https://github.com/PYangLab/M3.git" scanpy umap-learn

# Cell-level representation learning

This demo runs end-to-end on the **built-in Liu subsample** (~30k cells, 3 batches,
RNA + ADT) — one line to load. It uses the EXACT same parameters as the full-data
tutorial; only the data is subsampled. Use this to learn
the API and feel the workflow. For the full-data version that reproduces the
publication numbers, retrain on the full Liu dataset with the same parameters.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import m3

OUT = "./tutorial_out_demo/01_representation_learning"
os.makedirs(OUT, exist_ok=True)
sc.settings.verbosity = 1

## 1. Load the built-in demo (one line)

`m3.datasets.liu_demo()` returns a ready-to-go `Dataset`:
a stratified subsample of Liu et al. COVID-19.

In [ ]:
data = m3.datasets.liu_demo()
print(data)
print("celltypes:", data.obs["mergedcelltype"].value_counts().to_dict())

## 2. Build and train the integration model

Pure representation learning — no donor predictor. `batch_key="batch"` names the
obs column of batch labels (B1/B2/B3) that the VAE balances and corrects across.

In [ ]:
model = m3.M3(
    data,
    condition_keys=["cond_group", "Age_interval"],
    celltype_key="mergedcelltype",
    batch_key="batch",
    embedding_dim=30,
)
#To save time, users can set max_epochs to 100 for test.
model.train(max_epochs=500, donor_prediction=False)
print("capabilities:", model.capabilities)

## 3. Extract disentangled embeddings

In [ ]:
emb_bio = model.embedding(part="bio")
emb_batch = model.embedding(part="batch")
meta = model.cell_metadata.copy()
print("bio:", emb_bio.shape, "batch:", emb_batch.shape)

## 4. Save (npy / h5 / h5ad)

In [ ]:
import h5py
import anndata as ad

np.save(os.path.join(OUT, "embedding_bio.npy"), emb_bio)
meta.to_csv(os.path.join(OUT, "cell_metadata.csv"), index=False)
with h5py.File(os.path.join(OUT, "embedding_bio.h5"), "w") as f:
    f["data"] = emb_bio
ad.AnnData(X=emb_bio, obs=meta.astype(str).reset_index(drop=True)).write_h5ad(
    os.path.join(OUT, "embedding_bio.h5ad"))
print("saved -> .npy / .h5 / .h5ad in", OUT)

## 5. UMAP — bio + batch latent

In [ ]:
adn = sc.AnnData(X=emb_bio, obs=meta.reset_index(drop=True))
adn.obs["batch"] = adn.obs["batch"].astype("category")
adn.obs["mergedcelltype"] = adn.obs["mergedcelltype"].astype(str).astype("category")
adn.obs["cond_group"] = adn.obs["cond_group"].astype(str).astype("category")
sc.pp.neighbors(adn, use_rep="X", n_neighbors=15)
sc.tl.umap(adn)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key, title in zip(
    axes,
    ["mergedcelltype", "batch", "cond_group"],
    ["Cell type (biology preserved)", "Batch (mixed)", "Condition"],
):
    sc.pl.umap(adn, color=key, ax=ax, show=False, title=title,
               legend_loc="right margin" if key != "mergedcelltype" else "on data",
               legend_fontsize=6)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "umap_bio.png"), dpi=130, bbox_inches="tight")
plt.show()

adn_b = sc.AnnData(X=emb_batch, obs=meta.reset_index(drop=True))
adn_b.obs["batch"] = adn_b.obs["batch"].astype("category")
sc.pp.neighbors(adn_b, use_rep="X", n_neighbors=15)
sc.tl.umap(adn_b)
fig2, ax = plt.subplots(figsize=(6, 5))
sc.pl.umap(adn_b, color="batch", ax=ax, show=False, title="Batch latent — batch signal")
fig2.savefig(os.path.join(OUT, "umap_batch.png"), dpi=130, bbox_inches="tight")
plt.show()

**Done.** Integrated cell embedding ready for clustering / downstream analysis.
Retrain on the full Liu dataset for publication-scale results.